# Classificação Supervisionada com XGBoost

Além dos algoritmos anteriormente avaliados, foi realizado um experimento utilizando o XGBoost (Extreme Gradient Boosting), um algoritmo baseado em árvores de decisão que emprega a técnica de boosting para construir modelos mais robustos e capazes de capturar relações complexas entre os atributos.

A inclusão desse algoritmo teve como objetivo ampliar a comparação entre diferentes abordagens de aprendizado supervisionado, verificando se o XGBoost apresentaria desempenho superior ou complementar aos modelos já analisados, especialmente ao LightGBM, que havia obtido os melhores resultados até o momento.

Os experimentos foram realizados utilizando a mesma configuração definida nas etapas anteriores, incluindo o recorte temporal entre 2000 e 2008 e o novo esquema de rotulação com três classes: Adequada, Boa e Não adequada.

In [ ]:
!pip install xgboost

In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_PATH = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_2000_2008_novorotulo.parquet"
else:
    DATA_PATH = "../../dataset/processed/water_quality_2000_2008_novorotulo.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape: {df.shape}")
print(f"\nDistribuição do rótulo:")
print(df["conama_status"].value_counts())

Mounted at /content/drive
Dataset carregado com sucesso.
Shape: (59896, 23)

Distribuição do rótulo:
conama_status
Adequada        41169
Boa             11796
Não adequada     6931
Name: count, dtype: int64


## Preparação dos Dados

Para garantir a comparabilidade entre os algoritmos avaliados, foi utilizada a mesma base de dados empregada nos experimentos anteriores. Dessa forma, foram mantidas as mesmas variáveis preditoras, o mesmo conjunto de treinamento e teste e o mesmo esquema de codificação das classes.

As variáveis categóricas foram tratadas durante o pré-processamento por meio de codificação apropriada, permitindo que o modelo pudesse utilizar simultaneamente atributos numéricos e categóricos durante o treinamento.

In [ ]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes codificadas:")
for classe, codigo in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{classe}: {codigo}")

Classes codificadas:
Adequada: 0
Boa: 1
Não adequada: 2


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=SEED,
    stratify=y_encoded
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [ ]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

## Experimento 1: XGBoost sem Balanceamento

No primeiro experimento, o modelo foi treinado utilizando a distribuição original das classes presentes no conjunto de treinamento, sem aplicação de técnicas de balanceamento.

Esse cenário foi utilizado como linha de base para avaliar o comportamento natural do algoritmo diante do desbalanceamento existente na base de dados e servir como referência para os demais testes.

In [ ]:
model_sem_balanceamento = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

model_sem_balanceamento.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [ ]:
# MÉTRICAS DE TREINO sem balanceamento
y_train_pred = model_sem_balanceamento.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)

Train Accuracy:
0.7940980048418065
Train Precision:
0.7796157970849069
Train Recall:
0.7940980048418065
Train F1:
0.7797056796785374

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.88     32935
           1       0.67      0.44      0.53      9436
           2       0.63      0.53      0.58      5545

    accuracy                           0.79     47916
   macro avg       0.71      0.64      0.67     47916
weighted avg       0.78      0.79      0.78     47916

Train Confusion Matrix:
[[30907  1224   804]
 [ 4288  4182   966]
 [ 1784   800  2961]]


In [ ]:
# MÉTRICAS DE TESTE sem balanceamento
y_pred = model_sem_balanceamento.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.73889816360601

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.86      8234
           1       0.52      0.35      0.42      2360
           2       0.47      0.40      0.43      1386

    accuracy                           0.74     11980
   macro avg       0.60      0.55      0.57     11980
weighted avg       0.71      0.74      0.72     11980


Confusion Matrix:
[[7464  463  307]
 [1214  830  316]
 [ 513  315  558]]


## Experimento 2: XGBoost com Pesos Manuais

No segundo experimento, foram aplicados pesos definidos manualmente para cada classe durante o treinamento do modelo.

Essa estratégia teve como objetivo aumentar a importância das classes menos representadas, especialmente da classe Não adequada, reduzindo a tendência do algoritmo de favorecer a classe majoritária durante o processo de aprendizagem.

A definição dos pesos foi baseada nos testes realizados anteriormente com outros algoritmos de classificação, permitindo verificar se o XGBoost responderia de maneira semelhante às abordagens já avaliadas.

In [ ]:
# com balanceamento manual
class_weights = {
    0: 1,
    1: 2,
    2: 2
}

sample_weight_manual = np.array([
    class_weights[classe] for classe in y_train
])

model_peso_manual = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

model_peso_manual.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weight_manual
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [ ]:
# MÉTRICAS DE TREINO com balanceamento manual
y_train_pred = model_peso_manual.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)

Train Accuracy:
0.7805325987144169
Train Precision:
0.7913785867642665
Train Recall:
0.7805325987144169
Train F1:
0.7851317323480104

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.86      0.88     32935
           1       0.54      0.62      0.57      9436
           2       0.60      0.61      0.61      5545

    accuracy                           0.78     47916
   macro avg       0.68      0.70      0.69     47916
weighted avg       0.79      0.78      0.79     47916

Train Confusion Matrix:
[[28187  3450  1298]
 [ 2641  5805   990]
 [  626  1511  3408]]


In [ ]:
# MÉTRICAS DE TESTE com balanceamento manual
y_pred = model_peso_manual.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7171953255425709

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.83      0.85      8234
           1       0.43      0.50      0.46      2360
           2       0.43      0.43      0.43      1386

    accuracy                           0.72     11980
   macro avg       0.57      0.59      0.58     11980
weighted avg       0.73      0.72      0.72     11980


Confusion Matrix:
[[6815 1008  411]
 [ 783 1179  398]
 [ 217  571  598]]


## Experimento 3: XGBoost com Balanceamento Automático

No terceiro experimento, os pesos das classes foram calculados automaticamente a partir da distribuição dos dados de treinamento.

Nessa abordagem, classes com menor quantidade de exemplos recebem maior importância durante o treinamento, enquanto classes mais frequentes recebem pesos menores. O objetivo é reduzir os efeitos do desbalanceamento sem a necessidade de definição manual dos pesos.

Esse experimento permitiu comparar o desempenho do balanceamento automático com a estratégia de pesos manuais utilizada anteriormente.

In [ ]:
# com balanceamento automático
sample_weight_auto = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

model_balanceamento_auto = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

model_balanceamento_auto.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weight_auto
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [ ]:
# MÉTRICAS DE TREINO com balanceamento automático
y_train_pred = model_balanceamento_auto.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)

Train Accuracy:
0.7459512480173637
Train Precision:
0.7977918594024658
Train Recall:
0.7459512480173637
Train F1:
0.7594578094798365

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.79      0.85     32935
           1       0.57      0.52      0.54      9436
           2       0.43      0.87      0.58      5545

    accuracy                           0.75     47916
   macro avg       0.64      0.73      0.66     47916
weighted avg       0.80      0.75      0.76     47916

Train Confusion Matrix:
[[25978  3328  3629]
 [ 1835  4928  2673]
 [  261   447  4837]]


In [ ]:
# MÉTRICAS DE TESTE com balanceamento automático
y_pred = model_balanceamento_auto.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.6906510851419032

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.77      0.83      8234
           1       0.44      0.39      0.42      2360
           2       0.35      0.72      0.47      1386

    accuracy                           0.69     11980
   macro avg       0.57      0.63      0.57     11980
weighted avg       0.75      0.69      0.71     11980


Confusion Matrix:
[[6346  895  993]
 [ 575  932  853]
 [ 111  279  996]]


## Análise dos Resultados

Os experimentos realizados com o XGBoost demonstraram que diferentes estratégias de balanceamento influenciam diretamente o comportamento do modelo, principalmente na identificação da classe Não adequada.

No cenário sem balanceamento, o modelo apresentou o melhor desempenho geral, alcançando acurácia de 73,89% no conjunto de teste. Além disso, obteve a maior precisão para a classe Não adequada (47%), indicando menor incidência de falsos positivos quando comparado aos demais cenários. Entretanto, o recall dessa classe permaneceu limitado a 40%, indicando que parte dos casos críticos deixou de ser identificada.

A aplicação de pesos manuais produziu um comportamento mais equilibrado entre as classes. Embora a acurácia tenha reduzido para 71,72%, observou-se um aumento no recall da classe Não adequada para 43%, mantendo níveis de precisão semelhantes aos obtidos sem balanceamento. Esse resultado sugere que a estratégia manual conseguiu melhorar parcialmente a identificação dos casos de interesse sem comprometer excessivamente o desempenho geral do modelo.

Por sua vez, o balanceamento automático apresentou a maior capacidade de detecção da classe Não adequada, elevando o recall para 72%. Contudo, esse ganho ocorreu acompanhado de uma redução significativa na precisão (35%) e na acurácia geral (69,07%). Esse comportamento indica que o modelo passou a classificar um número muito maior de amostras como Não adequadas, aumentando consideravelmente a quantidade de falsos positivos.

De forma geral, os resultados mostram que o XGBoost apresentou desempenho competitivo em todos os cenários avaliados. O modelo sem balanceamento obteve o melhor desempenho global, enquanto o balanceamento automático priorizou a detecção da classe Não adequada. Já a estratégia de pesos manuais apresentou o melhor equilíbrio entre precisão, recall e acurácia, tornando-se a configuração mais adequada quando se busca conciliar desempenho geral e capacidade de identificação das situações críticas.
